# Kapitel 1 — Daten lesen · Notebook zum Ausfüllen

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch01_a_completer.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Die Zellen, die Sie bearbeiten sollen, sind mit **ZU ERGÄNZEN** markiert. Führen Sie die Zellen der Reihe nach von oben nach unten aus: Solange eine Antwort fehlt, melden die Prüfzellen dies, ohne einen Fehler auszulösen. Die Lösung finden Sie in `ch01_corrige.ipynb` auf derselben Begleitseite.

**Konvention.** Wie im Buch teilen Varianz, Standardabweichung, z-Wert und Kovarianz durch $n$ (`ddof=0`), sofern nicht ausdrücklich anders angegeben.

**Inhalt.** Daten · Die Berechnungen der Lektionen · Aufgaben 1.1 bis 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Farben des Buchs: eine Farbe = eine Rolle (beobachtet, vorhergesagt, Abweichung, Gerade des Kollegen)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Daten

Die Kennungen (D0.1, D1.1…) sind die des Buchs und der CSV-Dateien.

Variablen- und Spaltennamen im Code bleiben französisch: `logements` / `logement` = Wohnungen / Wohnung, `surface_m2` = Wohnfläche in m², `prix_kEUR` = Preis in k€, `nord` / `sud` = Viertel Nord / Süd, `registre` = Verkaufsregister, `vente` = Verkauf, `pieces` = Zimmer, `dpe` = Energieeffizienzklasse, `delai_jours` = Verkaufsdauer in Tagen, `etage` = Stockwerk, `delais_agences` = Verkaufsdauern der Agenturen, `tilleuls` / `port` = die beiden Wohnanlagen (Les Tilleuls, Le Port), `loyers` = Mieten, `moyenne` = Mittelwert, `ecart(s)` = Abweichung(en).

In [ ]:
# Daten von Kapitel 1, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D0.1 — die sechs Wohnungen der Einleitung
logements = pd.DataFrame({
    'logement':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'surface_m2': [32, 45, 54, 70, 85, 98],
    'prix_kEUR':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — Verkaufspreise der T2-Wohnungen (zwei Zimmer, k€), Lektion 1.1
nord = [174, 177, 181, 183, 185]
sud  = [150, 165, 185, 195, 205]

# D1.2 — Verkaufsregister des Viertels Süd, Lektionen 1.2 bis 1.5
registre = pd.DataFrame({
    'vente':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'pieces':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'surface_m2':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'dpe':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'prix_kEUR':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'delai_jours': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — Preis pro m² nach Stockwerk (k€/m²), Lektion 1.5
etages = pd.DataFrame({
    'etage':        [0, 1, 2, 3, 4, 5, 6],
    'prix_m2_kEUR': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — Verkaufsdauern von drei Agenturen (Tage), Aufgabe 1.1
delais_agences = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — Monatsmieten der Studios (€), Aufgabe 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

In [ ]:
# Bereitgestellte Werkzeuge: Sie müssen sie nicht ändern

def quartiles_livre(valeurs):
    """Q1 und Q3 nach der Regel des Buchs (Gleichung 1.5): Ränge ⌈n/4⌉ und ⌈3n/4⌉."""
    x = sorted(valeurs)
    n = len(x)
    rang_q1 = -(-n // 4)          # n/4 auf die nächste ganze Zahl aufgerundet
    rang_q3 = -(-3 * n // 4)      # 3n/4 auf die nächste ganze Zahl aufgerundet
    return x[rang_q1 - 1], x[rang_q3 - 1]   # Rang 1 = Position 0 in Python


def points_par_rangee(series, xlabel, bandes=None, ax=None):
    """Eine Punktreihe pro Datenreihe, wie in den Abbildungen von Lektion 1.1.
    bandes: Wörterbuch {Name: (Mittelwert, Standardabweichung)}, um Mittelwert ± Standardabweichung zu zeichnen."""
    if ax is None:
        fig, ax = plt.subplots()
    noms = list(series)
    couleurs = [BLEU, ORANGE, '#52514e']
    for i, nom in enumerate(noms):
        y = len(noms) - 1 - i
        if bandes and nom in bandes:
            m, s = bandes[nom]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[nom], [y] * len(series[nom]), s=50,
                   color=couleurs[i % 3], zorder=3)
    ax.set_yticks(range(len(noms)), list(reversed(noms)))
    ax.set_ylim(-0.6, len(noms) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Die Berechnungen der Lektionen, in Python

Diese Zellen wiederholen die Berechnungen der Lektionen 1.1 bis 1.5. Führen Sie sie aus und vergleichen Sie mit dem Buch. Hier werden `np.std` und `np.var` mit `ddof=0` aufgerufen, also mit einer Division durch $n$, wie im Kapitel.

In [ ]:
# Lektion 1.1 — Mittelwert, Varianz und Standardabweichung (Division durch n)
for nom, serie in [('Nord', nord), ('Süd', sud)]:
    print(f'{nom}: Mittelwert = {np.mean(serie):g} k€, Varianz = {np.var(serie, ddof=0):g} k€², '
          f'Standardabweichung = {np.std(serie, ddof=0):g} k€')

In [ ]:
# Lektion 1.2 — Häufigkeiten je Energieeffizienzklasse (ordinale qualitative Variable)
registre['dpe'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lektion 1.3 — Mittelwert, Median und Quartile der Verkaufsdauern
d = registre['delai_jours']
print('Mittelwert:', d.mean(), ' Median:', d.median())
print('ohne S13: Mittelwert', d[registre['vente'] != 'S13'].mean(),
      ' Median', d[registre['vente'] != 'S13'].median())
print('Quartile, Regel des Buchs (Gleichung 1.5):', quartiles_livre(d))
print('Quartile, NumPy-Standard (Interpolation):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lektion 1.4 — standardisierte Werte (Gleichung 1.6, Standardabweichung mit Division durch n)
for nom, serie, vente in [('Nord', nord, 188), ('Süd', sud, 210)]:
    m, s = np.mean(serie), np.std(serie, ddof=0)
    print(f'{nom}: z der Reihe = {np.round((np.array(serie) - m) / s, 4)}, '
          f'z des Verkaufs zu {vente} k€ = {(vente - m) / s:g}')

In [ ]:
# Lektion 1.5 — Kovarianz und Korrelation der fünf T2-Wohnungen im Süden (Division durch n)
t2 = registre[registre['pieces'] == 2]
x, y = t2['surface_m2'].to_numpy(float), t2['prix_kEUR'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('Kovarianz:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r mit np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r Stockwerk / Preis pro m²:', np.corrcoef(etages['etage'], etages['prix_m2_kEUR'])[0, 1].round(4))

## Aufgabe 1.1 — Ein Diagramm lesen

*Daten D1.E1. Begriffe: Mittelwert und Standardabweichung (Lektion 1.1).* Das folgende Diagramm gibt das aus Aufgabe 1.1 des Buchs wieder.

In [ ]:
points_par_rangee({f'Agentur {a}': v for a, v in delais_agences.items()},
                  'Verkaufsdauer (Tage)')
plt.xlim(0, 60)
plt.show()

**a) und b)** Schätzen Sie nach Augenmaß die mittlere Verkaufsdauer jeder Agentur und ordnen Sie die Agenturen dann von der am wenigsten zur am stärksten gestreuten.

In [ ]:
# ZU ERGÄNZEN: Ihre Schätzungen nach Augenmaß, vor jeder Rechnung
moyenne_estimee = {'P': None, 'Q': None, 'R': None}   # in Tagen
ordre_dispersion = None   # Liste, von der am wenigsten zur am stärksten gestreuten, z. B. ['Q', 'P', 'R']

In [ ]:
# Prüfung (Division durch n)
for a, v in delais_agences.items():
    print(f'Agentur {a}: Mittelwert = {np.mean(v):g} T, Standardabweichung = {np.std(v, ddof=0):.2f} T')
ordre_reel = sorted(delais_agences, key=lambda a: np.std(delais_agences[a]))
if ordre_dispersion is None or None in moyenne_estimee.values():
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    print('Reihenfolge der Streuung:', 'richtig' if ordre_dispersion == ordre_reel else 'bitte prüfen')

**c)** Zählen Sie für Q und für R die Verkäufe, die höchstens 2 Tage vom Mittelwert entfernt abgeschlossen wurden.

In [ ]:
# ZU ERGÄNZEN: Ihre Zählungen, aus dem Diagramm abgelesen
proches_Q = None
proches_R = None

In [ ]:
# Prüfung
for a, la_votre in [('Q', proches_Q), ('R', proches_R)]:
    v = np.array(delais_agences[a])
    vrai = int(np.sum(np.abs(v - v.mean()) <= 2))
    if la_votre is None:
        print(f'Agentur {a}: noch nicht ausgefüllt.')
    else:
        print(f"Agentur {a}: {'richtig' if la_votre == vrai else 'bitte prüfen'}")

*Ihre Erklärung zu c):* …

## Aufgabe 1.2 — Zwei Wohnanlagen, dieselbe mittlere Miete

*Daten D1.E2. Begriffe: Mittelwert, Varianz und Standardabweichung (Lektion 1.1).* Camille kann höchstens 570 € im Monat ausgeben. Die Fragen a) bis d) lösen Sie zuerst auf Papier, mit der Tabelle aus Aufgabe 1.2 des Buchs. Dieses Notebook dient dazu, Ihre Antworten zu prüfen, dann die Standardabweichung zu programmieren (Frage e) und zwei Konventionen zu vergleichen (Frage f).

In [ ]:
points_par_rangee({'Les Tilleuls': tilleuls, 'Le Port': port}, "Monatsmiete eines Studios (€)")
plt.axvline(570, color='black', linestyle=':', label='Budget von Camille: 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Die Mieten ablesen

Wie viele Studios kann Camille in jeder Wohnanlage mit 570 € mieten, und wie viele mit 620 €?

In [ ]:
# ZU ERGÄNZEN: Anzahl der bezahlbaren Studios, aus der Mietliste abgelesen
accessibles = {
    570: {'Les Tilleuls': None, 'Le Port': None},
    620: {'Les Tilleuls': None, 'Le Port': None},
}

In [ ]:
# Prüfung: `<=` liefert Wahr/Falsch-Werte, np.sum zählt die „Wahr“-Werte
for budget, reponses in accessibles.items():
    for nom, serie in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        vrai = int(np.sum(np.array(serie) <= budget))
        la_votre = reponses[nom]
        etat = 'noch nicht ausgefüllt' if la_votre is None else ('richtig' if la_votre == vrai else 'bitte prüfen')
        print(f'Budget {budget} €, {nom}: {etat}')

*Ihre Antwort zu a), in einem Satz:* …

### b) Die beiden Mittelwerte

Berechnen Sie sie auf Papier und tragen Sie Ihre Ergebnisse dann unten ein.

In [ ]:
# ZU ERGÄNZEN: Ihre von Hand berechneten Mittelwerte (in €)
moyenne_tilleuls_main = None
moyenne_port_main = None

In [ ]:
# Prüfung
for nom, serie, la_votre in [('Les Tilleuls', tilleuls, moyenne_tilleuls_main),
                              ('Le Port', port, moyenne_port_main)]:
    if la_votre is None:
        print(f'{nom}: bitte füllen Sie die vorherige Zelle aus.')
    else:
        ok = np.isclose(la_votre, np.mean(serie))
        print(f"{nom}: {'richtig' if ok else 'bitte prüfen'} (Ihr Wert: {la_votre} €)")

### c) Varianz und Standardabweichung von Hand (Division durch $n = 6$)

Erstellen Sie auf Papier für jede Wohnanlage die Tabelle der Abweichungen vom Mittelwert und ihrer Quadrate (Vorlage: die Tabelle der schrittweisen Berechnung in Lektion 1.1). Tragen Sie hier die vier Ergebnisse ein.

In [ ]:
# ZU ERGÄNZEN: Ihre von Hand berechneten Ergebnisse (Division durch n)
variance_tilleuls_main = None   # in €²
variance_port_main = None       # in €²
ecart_type_tilleuls_main = None # in €
ecart_type_port_main = None     # in €

In [ ]:
# Prüfung (Division durch n: ddof=0)
vos_valeurs = {
    'Varianz Tilleuls': (variance_tilleuls_main, np.var(tilleuls, ddof=0)),
    'Varianz Port': (variance_port_main, np.var(port, ddof=0)),
    'Standardabweichung Tilleuls': (ecart_type_tilleuls_main, np.std(tilleuls, ddof=0)),
    'Standardabweichung Port': (ecart_type_port_main, np.std(port, ddof=0)),
}
for nom, (la_votre, attendu) in vos_valeurs.items():
    if la_votre is None:
        print(f'{nom}: noch nicht ausgefüllt.')
    else:
        print(f"{nom}: {'richtig' if np.isclose(la_votre, attendu) else 'bitte prüfen'}")

### d) Interpretieren

*Ihre Antwort:* …

### e) Die Standardabweichung programmieren

Vervollständigen Sie die Funktion nach Gleichung 1.3 des Buchs, mit einer Division durch $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Verwenden Sie weder `np.std` noch `np.var`: Ziel ist, jeden Schritt selbst zu schreiben.

In [ ]:
# Aufgabe 1.2 e) — ZU ERGÄNZEN
def ecart_type(valeurs):
    """Standardabweichung, Division durch n (Gleichung 1.3)."""
    x = np.array(valeurs, dtype=float)
    n = len(x)
    moyenne = None   # ZU ERGÄNZEN: Gleichung 1.1
    ecarts = None    # ZU ERGÄNZEN: eine Abweichung pro Wert
    variance = None  # ZU ERGÄNZEN: Mittelwert der Quadrate
    return None      # ZU ERGÄNZEN: zurück zur ursprünglichen Einheit

<details><summary><b>Hinweis</b> (klicken Sie, wenn Sie nicht weiterkommen)</summary>

Mit NumPy wirkt eine auf ein Array geschriebene Operation auf jeden Wert: `x - moyenne` liefert auf einmal die sechs Abweichungen und `ecarts ** 2` ihre sechs Quadrate. `np.sum(...)` addiert, `np.sqrt(...)` zieht die Quadratwurzel.

</details>

In [ ]:
# Prüfung: Führen Sie diese Zelle aus, ohne sie zu ändern
if ecart_type(tilleuls) is None:
    print('Die Funktion gibt noch None zurück: Bitte vervollständigen Sie sie.')
else:
    for nom, serie, attendu in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtenu = ecart_type(serie)
        etat = 'richtig' if np.isclose(obtenu, attendu) else f'bitte prüfen (erwartet: {attendu})'
        print(f'{nom}: {obtenu:.2f} € → {etat}')

### f) Vertiefung: durch $n$ oder durch $n - 1$ teilen

Führen Sie die folgende Zelle aus. Finden Sie 10,95 von Hand, indem Sie die Summe der Quadrate durch $n - 1 = 5$ teilen. Welche pandas-Option liefert die Konvention des Kapitels?

*Ihre Antwort:* …

In [ ]:
loyers = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, Standard (Division durch n − 1):', loyers.std().round(2).to_dict())
print('pandas, ddof=0 (Division durch n)      :', loyers.std(ddof=0).round(2).to_dict())

## Aufgabe 1.3 — Ein überraschender Verkauf?

*Daten D1.1. Begriffe: standardisierter Wert (Lektion 1.4, Gleichung 1.6).* Eine T2-Wohnung (zwei Zimmer) wurde gerade für 196 k€ verkauft. Mittelwerte und Standardabweichungen: die aus Lektion 1.1, mit Division durch $n$.

In [ ]:
# ZU ERGÄNZEN: Ihre auf Papier durchgeführten Berechnungen
z_nord = None
z_sud = None
prix_sud_aussi_exceptionnel = None   # in k€, Frage c)

In [ ]:
# Prüfung (Standardabweichung mit Division durch n)
attendus = {
    'z Nord': (z_nord, (196 - np.mean(nord)) / np.std(nord, ddof=0)),
    'z Süd': (z_sud, (196 - np.mean(sud)) / np.std(sud, ddof=0)),
    'gleichwertiger Preis Süd': (prix_sud_aussi_exceptionnel,
                            np.mean(sud) + (196 - np.mean(nord)) / np.std(nord, ddof=0) * np.std(sud, ddof=0)),
}
for nom, (la_votre, vrai) in attendus.items():
    etat = 'noch nicht ausgefüllt' if la_votre is None else ('richtig' if np.isclose(la_votre, vrai) else 'bitte prüfen')
    print(f'{nom}: {etat}')

*Ihre Antwort zu b):* …

## Aufgabe 1.4 — Gehören Fläche und Preis zusammen?

*Daten D0.1 (die sechs Wohnungen der Einleitung). Begriffe: Kovarianz und Korrelation (Lektion 1.5, Gleichungen 1.7 und 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(logements['surface_m2'], logements['prix_kEUR'], color=BLEU)
for _, l in logements.iterrows():
    plt.annotate(l['logement'], (l['surface_m2'], l['prix_kEUR']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Wohnfläche (m²)')
plt.ylabel('Verkaufspreis (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Vor jeder Rechnung: Wird $r$ positiv oder negativ sein? Nahe bei 0, bei 0,5 oder bei 1?

*Ihre Antwort:* …

**b)** Vervollständigen Sie die folgenden Zeilen. Teilen Sie überall durch $n$.

In [ ]:
x = logements['surface_m2'].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
n = len(x)

produits = None     # ZU ERGÄNZEN: (x_i − x̄)(y_i − ȳ) für jede Wohnung
covariance = None   # ZU ERGÄNZEN: Gleichung 1.7, Division durch n
r = None            # ZU ERGÄNZEN: Gleichung 1.8 (Standardabweichungen mit ddof=0)

<details><summary><b>Hinweis</b></summary>

`x.mean()` liefert den Mittelwert, `x - x.mean()` die sechs Abweichungen. Das Produkt `*` zweier Arrays multipliziert elementweise. `x.std(ddof=0)` liefert die Standardabweichung mit Division durch $n$.

</details>

In [ ]:
# Prüfung
if r is None or covariance is None:
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    print('Kovarianz:', round(covariance, 2), 'm²·k€ →',
          'richtig' if np.isclose(covariance, 1637.5) else 'bitte prüfen (erwartet: 1637,5)')
    print('r:', round(r, 4), '→', 'richtig' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'bitte prüfen')

**c)** Eine Korrelation nahe 1, aber Abweichungen von 5 bis 11 k€ im Betrag: Ist das ein Widerspruch?

*Ihre Antwort:* …

In [ ]:
print('np.cov standardmäßig (Division durch n − 1):', np.cov(x, y)[0, 1])
print('np.cov mit bias=True (Division durch n)     :', np.cov(x, y, bias=True)[0, 1])

## Aufgabe 1.5 — Vorhersagen, dann prüfen

*Daten D1.1. Begriffe: Mittelwert und Standardabweichung (Lektion 1.1), Median und Interquartilsabstand (Lektion 1.3).* Eine sechste T2-Wohnung wird im Norden für 240 k€ verkauft.

**a)** Notieren Sie vor jeder Rechnung Ihre Vorhersagen: Steigen Mittelwert, Median, Standardabweichung und Interquartilsabstand stark, ein wenig oder gar nicht?

*Ihre Vorhersagen:* …

In [ ]:
nord_plus = nord + [240]
print('Nord:', sorted(nord))
print('Nord mit dem Verkauf zu 240 k€:', sorted(nord_plus))

**b)** Vervollständigen Sie die Funktion `resume`. Sie soll die vier Kennzahlen zurückgeben, mit der Standardabweichung durch $n$ geteilt und den Quartilen nach der Regel des Buchs berechnet (`quartiles_livre`, weiter oben bereitgestellt). Die Schlüssel des Wörterbuchs bleiben französisch: `moyenne` = Mittelwert, `médiane` = Median, `écart-type` = Standardabweichung, `écart interquartile` = Interquartilsabstand.

In [ ]:
# ZU ERGÄNZEN
def resume(serie):
    q1, q3 = quartiles_livre(serie)
    return {
        'moyenne': None,              # ZU ERGÄNZEN
        'médiane': None,              # ZU ERGÄNZEN: np.median
        'écart-type': None,           # ZU ERGÄNZEN: Division durch n
        'écart interquartile': None,  # ZU ERGÄNZEN: aus q1 und q3
    }

In [ ]:
tableau = pd.DataFrame({'Nord (5 Verk.)': resume(nord), 'Nord + 240 k€ (6 Verk.)': resume(nord_plus),
                        'Süd (5 Verk.)': resume(sud)})
if tableau.isna().any().any():
    print('Bitte vervollständigen Sie die Funktion resume.')
else:
    attendu = {'moyenne': (180, 190), 'médiane': (181, 182), 'écart-type': (4, 22.66),
               'écart interquartile': (6, 8)}
    for k, (avant, apres) in attendu.items():
        ok = np.isclose(tableau.loc[k].iloc[0], avant) and np.isclose(tableau.loc[k].iloc[1], apres, atol=0.005)
        print(f"{k}: {'richtig' if ok else 'bitte prüfen'}")
tableau.round(2)

In [ ]:
# Die beiden Reihen nach dem Vorbild der Abbildung „Mittelwert ± Standardabweichung“ aus Lektion 1.1
if tableau.isna().any().any():
    print('Bitte vervollständigen Sie die Funktion resume, um das Diagramm anzuzeigen.')
else:
    points_par_rangee({'Nord': nord, 'Nord + 240': nord_plus},
                      'Verkaufspreis einer T2-Wohnung (k€)',
                      bandes={'Nord': (np.mean(nord), np.std(nord, ddof=0)),
                              'Nord + 240': (np.mean(nord_plus), np.std(nord_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

**c)** Sind die Preise im Norden unvorhersehbarer geworden als die im Süden?

*Ihre Antwort:* …

## Aufgabe 1.6 — Herausforderung: zwei Geraden, derselbe mittlere Fehler

*Daten D0.1. Begriffe: vorzeichenbehaftete Abweichung und Betrag (Einleitung, Lektion 1.1), Standardabweichung (Lektion 1.1).* Nach Augenmaß gezeichnete Gerade: vorhergesagter Preis = 3 × Fläche + 30. Faule Gerade: vorhergesagter Preis = 222 k€ für alle Wohnungen.

In [ ]:
prix = logements['prix_kEUR'].to_numpy(dtype=float)
surface = logements['surface_m2'].to_numpy(dtype=float)

ecarts_oeil = None          # ZU ERGÄNZEN: beobachtet − vorhergesagt, nach Augenmaß gezeichnete Gerade
ecarts_paresseuse = None    # ZU ERGÄNZEN: beobachtet − vorhergesagt, faule Gerade

In [ ]:
# Die drei Kennzahlen aus Frage b) (Standardabweichung: Division durch n)
if ecarts_oeil is None or ecarts_paresseuse is None:
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    indicateurs = pd.DataFrame({
        nom: {'Mittel vorzeichenbeh. Abweichungen': e.mean(),
              'Mittel der Abweichungsbeträge': np.abs(e).mean(),
              'Std.-abw. vorzeichenbeh. Abweich.': e.std(ddof=0)}
        for nom, e in [('Gerade nach Augenmaß', ecarts_oeil), ('faule Gerade', ecarts_paresseuse)]
    })
    print('Abweichungen, Gerade nach Augenmaß:', ecarts_oeil)
    print('Abweichungen, faule Gerade        :', ecarts_paresseuse)
    print('Prüfung:', 'richtig' if np.allclose(indicateurs.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicateurs.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'bitte prüfen')
    display(indicateurs.round(2))

In [ ]:
if ecarts_oeil is not None and ecarts_paresseuse is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(surface - 0.8, ecarts_oeil, color=ORANGE, label="Gerade nach Augenmaß")
    ax.scatter(surface + 0.8, ecarts_paresseuse, color=BLEU, marker='s', label='faule Gerade')
    ax.set_xlabel('Wohnfläche (m²)')
    ax.set_ylabel('Abweichung beobachtet − vorhergesagt (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

*Ihre Antworten zu c) und d):* …

In [ ]:
# Kontrolle zu d): Die Standardabweichung der Abweichungen der faulen Geraden ist gleich der der Preise
if ecarts_paresseuse is not None:
    print(round(ecarts_paresseuse.std(ddof=0), 4), round(prix.std(ddof=0), 4))